# 第10章: 事前学習済み言語モデル（GPT型）

本章では、GPT型（Transformerのデコーダ型）の事前学習済みモデルを利用して、言語生成、評判分析器（ポジネガ分類器）の構築、ファインチューニング、強化学習などに取り組む。

## 90. 次単語予測

“The movie was full of"に続くトークン（トークン列ではなく一つのトークンであることに注意せよ）として適切なもの上位10個と、その確率（尤度）を求めよ。ただし、言語モデルへのプロンプトがどのようなトークン列に変換されたか、確認せよ。

In [1]:
# transformersライブラリをインストールします。
!pip install transformers

In [2]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

# 事前学習済みGPT-2モデルとトークナイザーをロード
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

# 入力プロンプト
text = "The movie was full of"

# プロンプトをトークン化し、その結果を表示
input_ids = tokenizer.encode(text, return_tensors='pt')
print(f"入力テキスト: '{text}'")
print(f"トークン化されたID: {input_ids}")
print(f"トークン列: {[tokenizer.decode(token_id) for token_id in input_ids[0]]}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

入力テキスト: 'The movie was full of'
トークン化されたID: tensor([[ 464, 3807,  373, 1336,  286]])
トークン列: ['The', ' movie', ' was', ' full', ' of']


上記の出力で、`input_ids`が言語モデルへのプロンプトが変換されたトークン列のIDです。`トークン列`はそれぞれのIDに対応するトークンです。

次に、このトークン列の次に来る単語を予測し、上位10個のトークンとその確率を計算します。

In [ ]:
# モデルにプロンプトを与えて次トークンの予測を取得
with torch.no_grad():
    outputs = model(input_ids)
    predictions = outputs.logits

# 最後のトークンの次の予測（ロジット）を取得
next_token_logits = predictions[0, -1, :]

# 確率に変換するためにsoftmaxを適用
probabilities = torch.softmax(next_token_logits, dim=-1)

# 確率の高い上位10個のトークンとその確率を取得
top_10_probabilities, top_10_indices = torch.topk(probabilities, 10)

print("\n次トークンの予測 (上位10個):")
for i, (prob, idx) in enumerate(zip(top_10_probabilities, top_10_indices)):
    token = tokenizer.decode(idx)
    print(f"{i+1}. トークン: '{token}', 確率: {prob.item():.4f}")



次トークンの予測 (上位10個):
1. トークン: ' jokes', 確率: 0.0219
2. トークン: ' great', 確率: 0.0186
3. トークン: ' laughs', 確率: 0.0115
4. トークン: ' bad', 確率: 0.0109
5. トークン: ' surprises', 確率: 0.0107
6. トークン: ' references', 確率: 0.0105
7. トークン: ' fun', 確率: 0.0100
8. トークン: ' humor', 確率: 0.0074
9. トークン: ' "', 確率: 0.0074
10. トークン: ' the', 確率: 0.0067


## 91. 続きのテキストの予測

“The movie was full of"に続くテキストを複数予測せよ。このとき、デコーディングの方法や温度パラメータ（temperature）を変えながら、予測される複数のテキストの変化を観察せよ。

In [ ]:
import torch

# 入力プロンプト
text = "The movie was full of"
input_ids = tokenizer.encode(text, return_tensors='pt')

print(f"入力プロンプト: '{text}'\n")

# 異なるtemperatureでテキストを生成
def generate_text_with_temperature(input_ids, model, tokenizer, temperature, num_sequences=3, max_length=50):
    print(f"--- temperature={temperature} の場合 ---")
    generated_ids = model.generate(
        input_ids,
        max_length=max_length,
        num_return_sequences=num_sequences,
        no_repeat_ngram_size=2,
        do_sample=True,
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id
    )

    for i, generated_id in enumerate(generated_ids):
        generated_text = tokenizer.decode(generated_id, skip_special_tokens=True)
        print(f"生成されたテキスト {i+1}: {generated_text}")
    print("\n")

# temperatureが低い場合 (より保守的で予測可能)
generate_text_with_temperature(input_ids, model, tokenizer, temperature=0.7)

# temperatureが中程度の場合
generate_text_with_temperature(input_ids, model, tokenizer, temperature=1.0)

# temperatureが高い場合 (より多様で創造的だが、一貫性が低くなる可能性あり)
generate_text_with_temperature(input_ids, model, tokenizer, temperature=1.5)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


入力プロンプト: 'The movie was full of'

--- temperature=0.7 の場合 ---
生成されたテキスト 1: The movie was full of hilarious moments in which the characters (played by actors and actresses), all of whom were in a state of shock, were completely oblivious to the fact that they were wearing a bikini.

Then the character of the guy who
生成されたテキスト 2: The movie was full of good jokes and good laughs. I had the time to watch some of the more serious comedies, but I can't say I liked it better since I don't think I've ever seen any of them. The fact that
生成されたテキスト 3: The movie was full of scenes from the '70s and '80s: the great, wonderful-looking cars of the 1950s, the cool-sounding V-8s of '60s. The cars were all over the place, so


--- temperature=1.0 の場合 ---
生成されたテキスト 1: The movie was full of jokes about black people being "wicked," or how blacks aren't black enough. "No!" he yelled. His character yelled, "It's not white people!"

"He gets it pretty hard."



生成されたテキスト 2: The movie was full of great perfor

## 92. 予測されたテキストの確率を計算

“The movie was full of"に続くテキストを予測し、生成された各単語の尤度を表示せよ（生成されるテキストが長いと出力が読みにくくなるので、適当な長さで生成を打ち切るとよい）。

In [2]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# 事前学習済みGPT-2モデルとトークナイザーをロード (もし未定義の場合)
# このセルが単独で実行された場合に備えて、ここでロードします。
if 'tokenizer' not in locals():
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
if 'model' not in locals():
    model = GPT2LMHeadModel.from_pretrained('gpt2')

# 入力プロンプト
text = "The movie was full of"
input_ids = tokenizer.encode(text, return_tensors='pt')

print(f"入力プロンプト: '{text}'\n")

# テキストを生成し、各ステップのロジットも取得
# max_new_tokensで生成長を制限して、出力が長くなりすぎないようにする
generated_output = model.generate(
    input_ids,
    max_new_tokens=20, # 適当な長さに設定
    num_return_sequences=1, # 1つのシーケンスを生成
    output_scores=True, # 各ステップのロジットを出力
    return_dict_in_generate=True, # 出力を辞書形式で返す
    do_sample=True, # サンプリングを有効にする
    temperature=0.7, # 温度を設定
    pad_token_id=tokenizer.eos_token_id
)

generated_ids = generated_output.sequences[0]
scores = generated_output.scores # 各ステップでの全トークンのロジット

# 生成されたトークンとプロンプトのトークンを分離
# generated_ids には入力プロンプトのトークンも含まれるため、プロンプト部分をスキップ
newly_generated_ids = generated_ids[input_ids.shape[-1]:]

print("生成されたテキストと各トークンの尤度:")

# 各生成されたトークンとその尤度を表示
for i, (token_id, score_tensor) in enumerate(zip(newly_generated_ids, scores)):
    # 現在のステップでの全トークンの確率を計算
    probabilities = torch.softmax(score_tensor, dim=-1)

    # 生成されたトークンの確率を取得
    token_probability = probabilities[0, token_id].item()

    # トークンをデコード
    token = tokenizer.decode(token_id)

    print(f"  トークン: '{token}', 確率: {token_probability:.4f}")

# 全体の生成されたテキストを表示
full_generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
print(f"\n全体で生成されたテキスト: {full_generated_text}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


入力プロンプト: 'The movie was full of'

生成されたテキストと各トークンの尤度:
  トークン: ' funny', 確率: 0.0189
  トークン: ' and', 確率: 0.2143
  トークン: ' heartbreaking', 確率: 0.0059
  トークン: ' moments', 確率: 0.7966
  トークン: '.', 確率: 0.2420
  トークン: ' The', 確率: 0.1382
  トークン: ' story', 確率: 0.0708
  トークン: ' was', 確率: 0.3343
  トークン: ' set', 確率: 0.0152
  トークン: ' in', 確率: 0.9023
  トークン: ' a', 確率: 0.3846
  トークン: ' world', 確率: 0.2834
  トークン: ' where', 確率: 0.8030
  トークン: ' violence', 確率: 0.0076
  トークン: ' has', 確率: 0.0563
  トークン: ' become', 確率: 0.3316
  トークン: ' a', 確率: 0.4369
  トークン: ' common', 確率: 0.0624
  トークン: ' occurrence', 確率: 0.3314
  トークン: '.', 確率: 0.3685

全体で生成されたテキスト: The movie was full of funny and heartbreaking moments. The story was set in a world where violence has become a common occurrence.


## 93. パープレキシティ

適当な文を準備して、事前学習済み言語モデルでパープレキシティを測定せよ。例えば、

+ The movie was full of surprises
+ The movies were full of surprises
+ The movie were full of surprises
+ The movies was full of surprises

の4文に対して、パープレキシティを測定して観察せよ（最後の2つの文は故意に文法的な間違いを入れた）。

In [3]:
import torch

# 問題90、91、92でロードしたtokenizerとmodelを使用します。
# もしこれらのオブジェクトがまだ定義されていない場合は、ここでロードします。
if 'tokenizer' not in globals():
    from transformers import GPT2LMHeadModel, GPT2Tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    model = GPT2LMHeadModel.from_pretrained('gpt2')

# 測定する文のリスト
sentences = [
    "The movie was full of surprises",
    "The movies were full of surprises",
    "The movie were full of surprises",
    "The movies was full of surprises"
]

print("パープレキシティの計算を開始します:\n")

for i, sentence in enumerate(sentences):
    # 文をトークン化し、入力IDとラベルIDを準備
    # ラベルは入力IDと同じで、モデルは次のトークンを予測するタスクとして扱います
    input_ids = tokenizer.encode(sentence, return_tensors='pt')
    labels = input_ids.clone()

    # モデルを評価モードに設定
    model.eval()

    with torch.no_grad():
        # モデルにフォワードパスを実行
        outputs = model(input_ids, labels=labels)

        # 損失（ネガティブ対数尤度）を取得
        # outputs.lossはすでに平均化されているか、適切な形で提供されます
        loss = outputs.loss

        # パープレキシティを計算
        perplexity = torch.exp(loss)

    print(f"文 {i+1}: '{sentence}'")
    print(f"  パープレキシティ: {perplexity.item():.4f}\n")

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


パープレキシティの計算を開始します:

文 1: 'The movie was full of surprises'
  パープレキシティ: 99.3537

文 2: 'The movies were full of surprises'
  パープレキシティ: 126.4822

文 3: 'The movie were full of surprises'
  パープレキシティ: 278.8782

文 4: 'The movies was full of surprises'
  パープレキシティ: 274.6611



## 94. チャットテンプレート

"What do you call a sweet eaten after dinner?"という問いかけに対する応答を生成するため、チャットテンプレートを適用し、言語モデルに与えるべきプロンプトを作成せよ。また、そのプロンプトに対する応答を生成し、表示せよ。

In [4]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# 問題90、91、92でロードしたtokenizerとmodelを使用します。
# もしこれらのオブジェクトがまだ定義されていない場合は、ここでロードします。
if 'tokenizer' not in globals():
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
if 'model' not in globals():
    model = GPT2LMHeadModel.from_pretrained('gpt2')

# メッセージを定義
messages = [
    {"role": "user", "content": "What do you call a sweet eaten after dinner?"}
]

# チャットテンプレートを適用してプロンプトを作成
# GPT-2は通常チャットテンプレートを直接サポートしていませんが、
# トークナイザーにチャットテンプレートが設定されている場合は利用できます。
# ここでは、GPT-2が直接的なチャットテンプレートを持たないため、一般的な形式に変換します。
# より高度なモデル（例: Llama, Mistral）ではtokenizer.apply_chat_templateがより効果的です。

# GPT-2のシンプルなプロンプト形式
# 実際には、特定のチャットテンプレートがトークナイザーに設定されている場合にのみtokenizer.apply_chat_templateが機能します。
# GPT-2の場合、通常は単にテキストを連結する形になります。
# ここでは簡略化のためにuserの質問をそのままプロンプトとします。
# もしtokenizerにchat_templateが設定されている場合は、以下のコードを使用できます。
# chat_template_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# 簡単なプロンプトを作成
chat_template_prompt = "ユーザー: " + messages[0]["content"] + "\nAI:"

print(f"言語モデルに与えるプロンプト:\n---\n{chat_template_prompt}\n---\n")

# プロンプトをトークン化
input_ids = tokenizer.encode(chat_template_prompt, return_tensors='pt')

# 応答を生成
generated_ids = model.generate(
    input_ids,
    max_new_tokens=50,  # 生成するトークンの最大長
    num_return_sequences=1,
    do_sample=True,
    temperature=0.7, # 適度な温度で生成
    pad_token_id=tokenizer.eos_token_id
)

# 生成されたテキストをデコード
response = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# プロンプト部分を除いた応答のみを取得
# GPT-2の性質上、応答全体にプロンプトが含まれるため、プロンプトの長さを考慮して切り取ります。
response_only = response[len(chat_template_prompt):].strip()

print(f"生成された応答: {response_only}")

言語モデルに与えるプロンプト:
---
ユーザー: What do you call a sweet eaten after dinner?
AI:
---

生成された応答: Well, I don't like that word, but I do like a sweet eaten.
AI: It's just that in the past, I had only eaten it once.
AI: Well, I guess that's a good thing.
AI


## 95. マルチターンのチャット

問題94で生成された応答に対して、追加で"Please give me the plural form of the word with its spelling in reverse order."と問いかけたときの応答を生成・表示せよ。また、その時に言語モデルに与えるプロンプトを確認せよ。

In [5]:
import torch

# 前回のメッセージ履歴を更新
# 問題94の response_only をアシスタントの返答として追加します
messages.append({"role": "assistant", "content": response_only})

# 新しいユーザーの質問を追加
new_question = "Please give me the plural form of the word with its spelling in reverse order."
messages.append({"role": "user", "content": new_question})

# GPT-2向けのマルチターンプロンプトの構築
# 履歴をすべて含める形式にします
multi_turn_prompt = ""
for msg in messages:
    role = "ユーザー" if msg["role"] == "user" else "AI"
    multi_turn_prompt += f"{role}: {msg['content']}\n"
multi_turn_prompt += "AI:"

print(f"マルチターンプロンプト:\n---\n{multi_turn_prompt}\n---\n")

# プロンプトをトークン化
input_ids = tokenizer.encode(multi_turn_prompt, return_tensors='pt')

# 応答を生成
generated_ids = model.generate(
    input_ids,
    max_new_tokens=50,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)

# デコードとプロンプト除去
full_response = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
new_response_only = full_response[len(multi_turn_prompt):].strip()

print(f"生成された新しい応答: {new_response_only}")

マルチターンプロンプト:
---
ユーザー: What do you call a sweet eaten after dinner?
AI: Well, I don't like that word, but I do like a sweet eaten.
AI: It's just that in the past, I had only eaten it once.
AI: Well, I guess that's a good thing.
AI
ユーザー: Please give me the plural form of the word with its spelling in reverse order.
AI:
---

生成された新しい応答: Well, I don't really know any better.
AI: Well, it's a nice-looking name.
AI: It's a good name.
AI: And it's really sweet.
AI: So sweet.
AI:


## 96. プロンプトによる感情分析

事前学習済み言語モデルで感情分析を行いたい。テキストを含むプロンプトを事前学習済み言語モデルに与え、（ファインチューニングは行わずに）テキストのポジネガを予測するという戦略で、[SST-2](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip)の開発データにおける正解率を測定せよ。

In [3]:
!pip install datasets

In [4]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset
from tqdm import tqdm

# モデルとトークナイザーの準備
if 'tokenizer' not in globals():
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
if 'model' not in globals():
    model = GPT2LMHeadModel.from_pretrained('gpt2')
model.eval()

# SST-2データセットのロード (trust_remote_codeを削除して修正)
try:
    dataset = load_dataset("nyu-mll/glue", "sst2")
except Exception as e:
    # 他のソースからのロードを試行
    dataset = load_dataset("nyu-mll/glue", "sst2")

dev_data = dataset['validation']

# ポジティブ・ネガティブに対応するトークンIDを取得
pos_id = tokenizer.encode("positive", add_special_tokens=False)[0]
neg_id = tokenizer.encode("negative", add_special_tokens=False)[0]

correct = 0
total = 100 # 時間短縮のため最初の100件で評価

print(f"{total}件のデータで感情分析を開始します...\n")

for i in tqdm(range(total)):
    item = dev_data[i]
    sentence = item['sentence']
    label = item['label'] # 1: positive, 0: negative

    # プロンプトの作成
    prompt = f"Sentence: {sentence}\nSentiment:"
    input_ids = tokenizer.encode(prompt, return_tensors='pt')

    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits[0, -1, :]

        # positive と negative のロジットを比較
        pos_score = logits[pos_id].item()
        neg_score = logits[neg_id].item()

        prediction = 1 if pos_score > neg_score else 0

        if prediction == label:
            correct += 1

accuracy = correct / total
print(f"\n評価完了")
print(f"正解率 (Accuracy): {accuracy:.4f}")

100件のデータで感情分析を開始します...



100%|██████████| 100/100 [00:50<00:00,  1.98it/s]


評価完了
正解率 (Accuracy): 0.5300


## 97. 埋め込みに基づく感情分析

事前学習済み言語モデルでテキストをベクトルで表現（エンコード）し、そのベクトルにフィードフォワード層を通すことで極性ラベルを予測するモデルを学習せよ。

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

# GPT-2でpaddingを血うためにpad_tokenを耭定する
tokenizer.pad_token = tokenizer.eos_token

# 1. データの準備: SST-2のテキストをベクトル（埋め込み）に変換
def extract_embeddings(data, model, tokenizer, device, limit=500):
    model.to(device)
    embeddings = []
    labels = []

    print(f"{limit}件のデータから埋め込みを抽出中...")
    for i in tqdm(range(min(len(data), limit))):
        item = data[i]
        text = item.get('sentence', item.get('text'))
        label = item['label']

        inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True).to(device)
        with torch.no_grad():
            outputs = model.transformer(inputs['input_ids'])
            # 最後のトークンの隠れ状態を文の表現として使用
            last_hidden_state = outputs.last_hidden_state[0, -1, :]

        embeddings.append(last_hidden_state.cpu())
        labels.append(label)

    return torch.stack(embeddings), torch.tensor(labels)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_data = dataset['train']

# 計算資源と時間の都合上、一部のデータを使用
X_train, y_train = extract_embeddings(train_data, model, tokenizer, device, limit=1000)
X_dev, y_dev = extract_embeddings(dev_data, model, tokenizer, device, limit=200)

# DataLoaderの作成
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
dev_loader = DataLoader(TensorDataset(X_dev, y_dev), batch_size=32)

1000件のデータから埋め込みを抽出中...


100%|██████████| 1000/1000 [03:22<00:00,  4.93it/s]


200件のデータから埋め込みを抽出中...


100%|██████████| 200/200 [00:34<00:00,  5.83it/s]


In [15]:
# 2. フィードフォワード層の定義
class SentimentClassifier(nn.Module):
    def __init__(self, input_dim):
        super(SentimentClassifier, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.fc(x)

# モデルの初期化
input_dim = X_train.shape[1]
classifier = SentimentClassifier(input_dim).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(classifier.parameters(), lr=1e-3)

# 3. 学習
print("学習を開始します...")
for epoch in range(10):
    classifier.train()
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device).float()
        optimizer.zero_grad()
        outputs = classifier(batch_X).squeeze()
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

# 4. 評価
classifier.eval()
correct_count = 0
total_count = 0
with torch.no_grad():
    for batch_X, batch_y in dev_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = classifier(batch_X).squeeze()
        predictions = (outputs > 0.5).long()
        correct_count += (predictions == batch_y).sum().item()
        total_count += batch_y.size(0)

print(f"\n埋め込みベースの分類器の正解率: {correct_count/total_count:.4f}")

学習を開始します...

埋め込みベースの分類器の正解率: 0.7050


## 98. ファインチューニング

問題96のプロンプトに対して、正解の感情ラベルをテキストの応答として返すように事前学習済みモデルをファインチューニングせよ。

In [20]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# 1. ファインチューニング用のデータ準備
def tokenize_function(examples):
    # GPT-2に学習させるためのプロンプト形式を作成します。
    # "Sentence: [文]\nSentiment: [positive/negative]" というテキストを生成します。
    texts = [f"Sentence: {s}\nSentiment: {'positive' if l == 1 else 'negative'}"
             for s, l in zip(examples['sentence'], examples['label'])]
    return tokenizer(texts, padding="max_length", truncation=True, max_length=64)

# トークナイザーにパディングトークンを設定（GPT-2のデフォルトでは設定されていないため）
tokenizer.pad_token = tokenizer.eos_token

# 学習時間を短縮するため、データセットの一部（学習500件、検証100件）を抽出してトークナイズします
train_dataset = dataset['train'].select(range(500)).map(tokenize_function, batched=True)
eval_dataset = dataset['validation'].select(range(100)).map(tokenize_function, batched=True)

# 2. 学習（TrainingArguments）の設定
training_args = TrainingArguments(
    output_dir="./gpt2-sentiment-finetuned",
    eval_strategy="epoch",      # エポックごとに評価を実施
    learning_rate=5e-5,          # 学習率
    num_train_epochs=3,          # エポック数
    per_device_train_batch_size=4, # バッチサイズ
    save_total_limit=1,          # 保存するチェックポイントの数を制限
    logging_steps=20,            # 20ステップごとにログを出力
    report_to="none"             # 外部ツールへのレポートを無効化
)

# 言語モデル用のデータコレーター（入力文を1トークンずらしてラベルを自動生成する）
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 3. Trainerの初期化と学習の実行
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

print("GPT-2の感情分析用ファインチューニングを開始します...")
trainer.train()
print("\n学習が正常に完了しました。")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

GPT-2の感情分析用ファインチューニングを開始します...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,2.137872,3.345173
2,1.597727,3.667635
3,1.439766,3.809892


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



学習が正常に完了しました。


## 99. 選好チューニング

問題96のプロンプトに対して、正解の感情ラベルを含むテキストを望ましい応答、間違った感情ラベルを含むテキストを望ましくない応答として、事前学習済み言語モデルを選好チューニング (preference tuning) を実施せよ。選好チューニングのアルゴリズムとしては、近傍方策最適化 (PPO: Proximal Policy Optimization) や直接選好最適化 (DPO: Direct Preference Optimization) などが考えられる。


In [12]:
!pip install trl

import torch
from datasets import Dataset
from trl import DPOConfig, DPOTrainer

# 1. DPO用のデータセット作成
def prepare_dpo_dataset(data, limit=200):
    dpo_data = {
        "prompt": [],
        "chosen": [],
        "rejected": []
    }
    for i in range(min(len(data), limit)):
        item = data[i]
        sentence = item['sentence']
        label = item['label']

        dpo_data["prompt"].append(f"Sentence: {sentence}\nSentiment:")
        if label == 1:
            dpo_data["chosen"].append(" positive")
            dpo_data["rejected"].append(" negative")
        else:
            dpo_data["chosen"].append(" negative")
            dpo_data["rejected"].append(" positive")

    return Dataset.from_dict(dpo_data)

dpo_train_dataset = prepare_dpo_dataset(dataset['train'], limit=200)
dpo_eval_dataset = prepare_dpo_dataset(dataset['validation'], limit=50)

# 2. DPOの設定 (trl 1.5.1 CPU互換構成)
dpo_config = DPOConfig(
    output_dir="./gpt2-dpo",
    beta=0.1,
    eval_strategy="steps",
    eval_steps=20,
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=10,
    report_to="none",
    fp16=False,
    bf16=False,
    use_cpu=True
)

# 3. DPOTrainerの初期化と実行
# 注意: trl 1.5.1以降、DPOTrainerに直接tokenizerを渡す引数は削除されました。
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_train_dataset,
    eval_dataset=dpo_eval_dataset
)

print("DPOによる選好チューニングを開始します（CPU/trl 1.5.1互換）...")
dpo_trainer.train()
print("\n選好チューニングが完了しました。")

Adding EOS to train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


DPOによる選好チューニングを開始します（CPU/trl 1.5.1互換）...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
20,0.655821,0.651136,6.233623,1688.000000,-108.429340,-107.797934,0.000000,-0.292560,-0.381307,0.767857,0.088748,-14.617705,-15.861890
40,0.597550,0.610750,5.479415,3546.000000,-103.785056,-103.441971,0.000000,-0.108253,-0.314909,0.767857,0.206656,-12.774641,-15.197906
60,0.523514,0.587704,5.074776,5260.000000,-103.937186,-103.625666,0.000000,-0.004301,-0.308316,0.767857,0.304015,-11.735123,-15.131984
80,0.518719,0.566899,4.697516,6904.000000,-105.376338,-105.027425,0.000000,0.067684,-0.340577,0.767857,0.408261,-11.015265,-15.454585
100,0.498417,0.499686,4.560555,8694.000000,-104.343658,-103.903998,0.008929,0.119640,-0.461140,0.857143,0.580780,-10.495705,-16.660217


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


選好チューニングが完了しました。
